# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, understand, and analyze the FAIR² protein dataset describing protein abundance and modifications in human (Homo sapiens) samples, utilizing the [mlcroissant](https://mlcommons.org/croissant/) library for handling [Croissant](https://mlcommons.org/croissant/) schema-defined datasets.

### Dataset Source
This dataset is accessed via a Croissant schema URL and is described according to the FAIR² principles.

In [ ]:
# Install mlcroissant if not already available
!pip install mlcroissant --quiet

## 1. Data Loading
Load dataset metadata and explore main information using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Read the dataset (metadata & accessors)
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Access as object

print(f"Dataset title: {metadata.name}")
print(f"Published: {metadata.datePublished}")
print(f"Description: {metadata.description}")
print(f"Version: {metadata.version}\n")
print(f"Cite as: {metadata.citeAs}")

## 2. Data Overview
List the available record sets and their field `@id`s.

In [ ]:
# Enumerate available record sets
record_sets = list(metadata.recordSets)
print(f"Number of record sets: {len(record_sets)}\n")
for record_set in record_sets:
    print(f"RecordSet Name: {record_set.name}, @id: {record_set.id}")
    if hasattr(record_set, "fields"):
        print("  Fields (by @id):")
        for field in record_set.fields:
            print(f"    - {field.name:25} (@id: {field.id})")
    print()
    # List columns as well if present (for CSV/tabular)
    if hasattr(record_set, "columns"):
        print("  Columns (by @id):")
        for column in record_set.columns:
            print(f"    - {column.name:25} (@id: {column.id})")
        print()

## 3. Data Extraction
Load data from a specific record set into a pandas DataFrame for analysis. All record sets and their `@id`s should be listed, and use those `@id`s to extract the data.

In [ ]:
# Prepare mapping from record set names to IDs
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}
print(f"Record set IDs found: {record_set_ids}")

# Extract data from each record set
for rs_id in record_set_ids:
    # Load records using their Croissant @id
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded DataFrame for record set '@id': {rs_id}, shape: {df.shape}")
            print(f"Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Failed to load record set {rs_id}: {e}")

# Choose one record set (first) for demonstration. Replace below with target if needed.
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    df = dataframes[main_record_set_id]
    print(f"\nFirst 5 rows of record set '@id': {main_record_set_id}")
    display(df.head())
else:
    main_record_set_id = None
    print("No tabular record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply some basic data processing: filtering, normalization, grouping. Choose a numeric field using its `@id` as shown in the overview.

In [ ]:
# Pick a numeric field for demonstration (you may change this to e.g. 'cr:field/abundance' or similar by inspecting record set columns)
import numpy as np

# Find first numeric column if possible
numeric_field = None
if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    # Try to select the first numeric column by dtype
    for col in df.columns:
        if np.issubdtype(df[col].dropna().__class__, np.number):
            numeric_field = col
            break
    # Or look for plausible column names
    for col in df.columns:
        if any(word in col.lower() for word in ["abundance", "quantity", "mw", "weight", "peptide", "intensity"]):
            numeric_field = col
            break
    
if numeric_field:
    threshold = df[numeric_field].median()  # Use median for demonstration
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Records with '{numeric_field}' > {threshold} (using as threshold):")
    display(filtered_df.head())
    # Normalize (z-score)
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} (first 5):")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Try to group by a categorical field (e.g. 'cr:field/sample' or 'SampleID' if present)
    group_field = None
    for col in df.columns:
        if any(word in col.lower() for word in ["sample", "group", "type", "class"]):
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean {numeric_field} by '{group_field}':")
        display(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Display data distributions or relationships between fields using matplotlib or seaborn.

Replace field names and groupings if appropriate for your target analysis.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_field:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    # If grouping field found, plot grouped mean
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(10,4))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
We demonstrated how to:
- Load a FAIR² Croissant dataset using the `mlcroissant` library
- Explore metadata, record sets, and fields using their Croissant `@id`s
- Extract and analyze protein data with pandas
- Apply basic EDA and visualization

You can use the record set and field `@id` structure for robust, reusable data access, and extend this workflow for more complex downstream analysis or machine learning pipelines.

***
For more information about Croissant schemas and tools, visit [mlcommons.org/croissant](https://mlcommons.org/croissant/).